# Week 5: Northernmost, Easternmost, and Windiest

**Sebastian Perdomo | IS 362**

Data comes from the nycflights13 CSV files: `airports.csv` and `weather.csv`.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np

base = "https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/"
airports = pd.read_csv(base + "airports.csv")
weather = pd.read_csv(base + "weather.csv")

cols = ["faa", "name", "lat", "lon", "tz", "tzone"]
print(airports.shape, weather.shape)

(1458, 8) (26115, 15)


## 2. Northernmost airport

Top five airports by latitude, before any changes.

In [2]:
airports.nlargest(5, "lat")[cols]

,faa,name,lat,lon,tz,tzone
417,EEN,Dillant Hopkins Airport,72.270833,42.898333,-5,NaN
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,-9,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,-9,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,-9,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,-9,America/Anchorage


Dillant-Hopkins Airport (EEN) is in Keene, New Hampshire, not the Arctic. Its
latitude and longitude are swapped and the longitude lost its minus sign. The
correct coordinates are 42.898333, -72.270833.

In [3]:
airports.loc[airports["faa"] == "EEN", ["lat", "lon"]] = [42.898333, -72.270833]
airports.loc[airports["faa"] == "EEN", "tzone"] = "America/New_York"

airports.nlargest(5, "lat")[cols]

,faa,name,lat,lon,tz,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,-9,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,-9,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,-9,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,-9,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,-9,America/Anchorage


**Answer:** Wiley Post-Will Rogers Memorial Airport (BRW) in Utqiagvik, Alaska, at
71.29°N is the northernmost airport in the United States.

## 3. Easternmost airport

Top five airports by longitude, before any changes.

In [4]:
airports.nlargest(5, "lon")[cols]

,faa,name,lat,lon,tz,tzone
1290,SYA,Eareckson As,52.712275,174.113620,-9,America/Anchorage
942,MYF,Montgomery Field,32.475900,117.759000,8,Asia/Chongqing
396,DVT,Deer Valley Municipal Airport,33.411700,112.457000,8,Asia/Chongqing
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,-5,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,-5,America/New_York


Two of these have bad coordinates. Deer Valley (DVT) is in Phoenix and Montgomery
Field (MYF) is in San Diego, but both show positive longitudes and a China time
zone. I replaced them with their actual coordinates.

Eareckson Air Station sits on Shemya Island in the Aleutians,
past the 180° meridian, so its longitude is positive. That makes it "east" only on
paper, since it is actually one of the westernmost points in the country. To get
the easternmost airport in the usual sense, I shift any positive longitude by
-360 so the whole country is measured west from Greenwich.

In [5]:
airports.loc[airports["faa"] == "DVT", ["lat", "lon", "tz", "tzone"]] = [33.688306, -112.082556, -7, "America/Phoenix"]
airports.loc[airports["faa"] == "MYF", ["lat", "lon", "tz", "tzone"]] = [32.815833, -117.139444, -8, "America/Los_Angeles"]

airports["lon_west"] = np.where(airports["lon"] > 0, airports["lon"] - 360, airports["lon"])

airports.nlargest(5, "lon_west")[cols]

,faa,name,lat,lon,tz,tzone
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,-5,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,-5,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,-5,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,-5,America/New_York
1398,WFK,Northern Aroostook Regional Airport,47.285556,-68.312778,-5,America/New_York


**Answer:** Eastport Municipal Airport (EPM) in Eastport, Maine, at 67.01°W.
Eastport is the easternmost city in the United States.

## 4. Windiest New York area airport on February 12, 2013

Average and maximum wind speed (mph) for each airport that day, before any changes.

In [2]:
feb12 = weather[(weather["month"] == 2) & (weather["day"] == 12)].copy()
feb12.groupby("origin")["wind_speed"].agg(["mean", "max"]).round(2)

,mean,max
origin,,
EWR,56.39,1048.36
JFK,14.38,20.71
LGA,14.96,23.02


EWR shows a maximum of 1,048 mph. That is impossible. The next-highest reading in the whole file
is 42.6 mph. I treat it as a data error and set it to missing.

In [7]:
feb12.loc[feb12["wind_speed"] > 100, "wind_speed"] = np.nan

feb12.groupby("origin")[["wind_speed", "wind_gust"]].agg(["mean", "max"]).round(2)

wind_speed        wind_gust       
             mean    max      mean    max
origin                                   
EWR         13.26  21.86     25.17  31.07
JFK         14.38  20.71     24.58  27.62
LGA         14.96  23.02     24.89  31.07

**Answer:** LaGuardia (LGA) airport had the windiest weather. It had the highest average wind speed, 14.96 mph,
and the highest sustained wind of the day, 23.02 mph. Without removing the bad
reading, EWR would have looked like the answer.

## Assumptions and changes to the data

**Assumptions**
- Easternmost means farthest east measured west from Greenwich. Airports past
  the 180° meridian are counted as far west.
- Windiest means the highest average wind speed over the day.

**Changes made**

| Airport | Problem | Change | Source |
|---|---|---|---|
| EEN, Keene NH | Latitude and longitude swapped, longitude sign missing | Set to 42.898333, -72.270833 | [Wikipedia](https://en.wikipedia.org/wiki/Dillant%E2%80%93Hopkins_Airport) |
| DVT, Phoenix AZ | Wrong coordinates and time zone | Set to 33.688306, -112.082556, America/Phoenix | [Wikipedia](https://en.wikipedia.org/wiki/Phoenix_Deer_Valley_Airport) |
| MYF, San Diego CA | Wrong coordinates and time zone | Set to 32.815833, -117.139444, America/Los_Angeles | [Wikipedia](https://en.wikipedia.org/wiki/Montgomery-Gibbs_Executive_Airport) |
| EWR, Feb 12 3am | Wind speed of 1,048 mph | Set to missing | Outlier |

**Answers confirmed with**
- Northernmost: [Wiley Post-Will Rogers Memorial Airport (Wikipedia)](https://en.wikipedia.org/wiki/Wiley_Post%E2%80%93Will_Rogers_Memorial_Airport)
- Easternmost: [Eastport, Maine (Wikipedia)](https://en.wikipedia.org/wiki/Eastport,_Maine)
- SYA longitude: [Eareckson Air Station (Wikipedia)](https://en.wikipedia.org/wiki/Eareckson_Air_Station)